# constrained_checkpoint_selector_v1

Purpose: turn the method PK decision into a reusable checkpoint-selection gate. The selector must rank candidates by hard walk-forward constraints before objective/alignment scores, using the current target of excess return >= +30pp and MaxDD <= 15% on every evaluated window.

Source context:
- `external_method_pk_overfit_bottleneck_v1` selected a co-primary route: reward short-training screen generates a candidate, constrained checkpoint selector gates it.
- `short_volshock_bc_checkpoint_alignment_v1` provides current candidate official warm-start metrics and checkpoint objective scores.
- `short_volshock_drawdown_drift_reward_recipe_screen_v1` provides the reward recipe allowed to create a short-training candidate.

External basis retained from PK: Qlib/FinRL workflow discipline plus constrained-policy objective literature. No actor/profile promotion happens in this notebook.

In [1]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
import sys

WORKDIR = Path.cwd().resolve()
for candidate_root in (WORKDIR, *WORKDIR.parents):
    if (candidate_root / "AGENTS.md").exists():
        REPO_ROOT = candidate_root
        break
else:
    raise RuntimeError("Could not locate repository root from notebook working directory")
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

from src.soxl_quant.app.constrained_checkpoint_selector import (
    ConstrainedCheckpointSelectorConfig,
    render_constrained_checkpoint_selection_review,
    score_constrained_checkpoints,
    summarize_constrained_checkpoint_selection,
)

CARD = REPO_ROOT / "runtime/model_selection/card/constrained_checkpoint_selector_v1"
SOURCE_ALIGNMENT = REPO_ROOT / "runtime/model_selection/card/short_volshock_bc_checkpoint_alignment_v1"
SOURCE_FEATURE_AUDIT = REPO_ROOT / "runtime/model_selection/card/short_volshock_drawdown_drift_feature_audit_v1"
SOURCE_REWARD = REPO_ROOT / "runtime/reward/card/short_volshock_drawdown_drift_reward_recipe_screen_v1"
SOURCE_PK = REPO_ROOT / "runtime/model_selection/card/external_method_pk_overfit_bottleneck_v1"
CARD.mkdir(parents=True, exist_ok=True)

def read_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as fh:
        return json.load(fh)

def json_default(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

In [2]:
warm_metrics = pd.read_csv(SOURCE_ALIGNMENT / "warm_start_official_metrics.csv")
checkpoint_scores = pd.read_csv(SOURCE_ALIGNMENT / "checkpoint_objective_scores.csv")
gate_gaps = pd.read_csv(SOURCE_FEATURE_AUDIT / "gate_gap_attribution.csv")
reward_summary = read_json(SOURCE_REWARD / "summary.json")
pk_summary = read_json(SOURCE_PK / "summary.json")

gate_metrics = warm_metrics[[
    "candidate",
    "fold",
    "official_warm_relative_return",
    "official_warm_max_dd",
]].copy()

print("gate rows", len(gate_metrics), "candidates", sorted(gate_metrics["candidate"].unique()))
print("reward decision", reward_summary.get("decision"))
print("pk primary/secondary", pk_summary.get("primary_next"), pk_summary.get("secondary_next"))

gate rows 6 candidates ['base_multi_obj_plain', 'short_volshock_plain', 'short_volshock_weighted_obj025_cb099']
reward decision drawdown_drift_reward_recipe_ready_for_short_training_screen
pk primary/secondary constrained_checkpoint_selector reward_constrained_short_training_screen


In [3]:
config = ConstrainedCheckpointSelectorConfig(
    target_relative_return_pp=30.0,
    target_max_dd_pct=15.0,
    max_dd_gap_weight=10.0,
)

scored = score_constrained_checkpoints(
    gate_metrics,
    checkpoint_scores=checkpoint_scores,
    reference_candidate="base_multi_obj_plain",
    config=config,
)
summary = summarize_constrained_checkpoint_selection(scored)
review = render_constrained_checkpoint_selection_review(scored, summary)

scored

,candidate,selector_gate_pass,decision,constrained_selector_loss,constraint_gap_loss,failed_window_loss,drift_loss,score_loss,window_count,gate_pass_count,...,min_relative_return,max_max_dd,relative_drift_gap,max_dd_drift_gap,alignment_loss,checkpoint_objective_loss,missing_target_ratio,coverage_gate,target_relative_return_pp,target_max_dd_pct
0,short_volshock_plain,False,constrained_checkpoint_no_promotion,322.434672,114.531866,200.0,7.185089,0.717717,2,0,...,5.018899,22.261989,28.740358,0.0,0.151457,2.265041,0.0,True,30.0,15.0
1,short_volshock_weighted_obj025_cb099,False,constrained_checkpoint_no_promotion,325.685832,118.636013,200.0,6.234051,0.815768,2,0,...,8.823051,24.745906,24.936205,0.0,0.155451,2.641265,0.0,True,30.0,15.0
2,base_multi_obj_plain,False,constrained_checkpoint_no_promotion,604.181338,272.392601,200.0,0.000000,131.788737,2,0,...,33.759257,36.332029,0.000000,0.0,2.237350,18.205547,1.0,False,30.0,15.0


In [4]:
sidecars = {
    "scores": CARD / "constrained_checkpoint_scores.csv",
    "selection": CARD / "selection.json",
    "summary": CARD / "summary.json",
    "review": CARD / "constrained_checkpoint_review.md",
}

selection = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "experiment": "constrained_checkpoint_selector_v1",
    "decision": summary["decision"],
    "selected_candidate": summary["selected_candidate"],
    "selector_gate_pass": summary["selector_gate_pass"],
    "promotion_allowed": summary["promotion_allowed"],
    "replay_review_required": summary["replay_review_required"],
    "next_mechanism": summary["next_mechanism"],
    "acceptance_criteria": summary["acceptance_criteria"],
    "source_context": {
        "method_pk_decision": pk_summary.get("decision"),
        "method_pk_primary_next": pk_summary.get("primary_next"),
        "method_pk_secondary_next": pk_summary.get("secondary_next"),
        "reward_recipe_decision": reward_summary.get("decision"),
        "reward_training_allowed_next": reward_summary.get("training_allowed_next"),
        "alignment_decision": read_json(SOURCE_ALIGNMENT / "checkpoint_summary.json").get("decision"),
    },
    "scheme_pk_adjustment": (
        "Keep the co-primary route, but clarify order: reward short-training creates a new candidate; "
        "constrained selector gates promotion/replay. Current BC candidates remain blocked."
    ),
    "external_sources": pk_summary.get("external_sources", []),
    "sidecars": [path.name for path in sidecars.values()],
}

scored.to_csv(sidecars["scores"], index=False)
with sidecars["selection"].open("w", encoding="utf-8") as fh:
    json.dump(selection, fh, indent=2, sort_keys=True, default=json_default)
summary_out = dict(summary)
summary_out.update({
    "experiment": "constrained_checkpoint_selector_v1",
    "created_at": selection["created_at"],
    "source_cards": [
        "short_volshock_bc_checkpoint_alignment_v1",
        "short_volshock_drawdown_drift_feature_audit_v1",
        "short_volshock_drawdown_drift_reward_recipe_screen_v1",
        "external_method_pk_overfit_bottleneck_v1",
    ],
})
with sidecars["summary"].open("w", encoding="utf-8") as fh:
    json.dump(summary_out, fh, indent=2, sort_keys=True, default=json_default)
sidecars["review"].write_text(review, encoding="utf-8")

summary_out

{'decision': 'constrained_checkpoint_selector_no_promotion',
 'selected_candidate': 'short_volshock_plain',
 'selected_constrained_selector_loss': 322.4346721245639,
 'selected_constraint_gap_loss': 114.53186580251739,
 'selected_failed_window_count': 2,
 'selected_relative_failure_count': 1,
 'selected_maxdd_failure_count': 2,
 'selected_min_relative_return': 5.018898654205939,
 'selected_max_max_dd': 22.26198899445525,
 'selector_gate_pass': False,
 'promotion_allowed': False,
 'replay_review_required': False,
 'next_mechanism': 'generate_reward_constrained_short_training_candidate',
 'acceptance_criteria': {'target_relative_return_pp': 30.0,
  'target_max_dd_pct': 15.0,
  'all_windows_must_pass': True},
 'experiment': 'constrained_checkpoint_selector_v1',
 'created_at': '2026-05-26T15:20:03.229920+00:00',
 'source_cards': ['short_volshock_bc_checkpoint_alignment_v1',
  'short_volshock_drawdown_drift_feature_audit_v1',
  'short_volshock_drawdown_drift_reward_recipe_screen_v1',
  'ext

## Runtime conclusion

The current candidate set does not contain a promotable checkpoint. `short_volshock_plain` is the closest diagnostic under the constrained selector, but it fails both evaluated windows: 2022 breaches MaxDD and 2023/2024 misses the +30pp excess target while also breaching MaxDD. `short_volshock_weighted_obj025_cb099` fixes the 2023/2024 MaxDD side but still misses 2023/2024 excess return and worsens 2022 MaxDD.

PK adjustment: no replacement of the co-primary plan is needed. The operational order is now explicit: use the reward-constrained short-training screen to generate a new candidate, then apply this constrained selector before any replay/promotion discussion. The selector is also the guardrail for future network/model/reward attempts so validation score cannot outrank the hard +30pp / <=15% walk-forward gate.